# Apriori Algorithm Implementation Assignment

### Objective:
You will implement the **Apriori algorithm** from scratch (i.e., without using any libraries like `mlxtend`) to find frequent itemsets and generate association rules.

### Dataset:
Use the [Online Retail Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail) from Kaggle. You can filter it for a specific country (e.g., `United Kingdom`) and time range to reduce size if needed.

---

## Step 1: Data Preprocessing

- Load the dataset
- Remove rows with missing values
- Filter out rows where `Quantity <= 0`
- Convert Data into Basket Format

👉 **Implement code below**

In [10]:
# Load the dataset
import pandas as pd
import numpy as np

# Read CSV file
data = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')

# Preprocess
data = data.dropna()   
data = data[data['Quantity'] > 0]   

# Create Basket
basket = (
    data
    .groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack()
    .reset_index()
    .fillna(0)
    .set_index('InvoiceNo')
)
basket = basket.map(lambda x: 1 if x > 0 else 0)
print(basket.head())


Description   4 PURPLE FLOCK DINNER CANDLES   50'S CHRISTMAS GIFT BAG LARGE  \
InvoiceNo                                                                     
536365                                    0                               0   
536366                                    0                               0   
536367                                    0                               0   
536368                                    0                               0   
536369                                    0                               0   

Description   DOLLY GIRL BEAKER   I LOVE LONDON MINI BACKPACK  \
InvoiceNo                                                       
536365                        0                             0   
536366                        0                             0   
536367                        0                             0   
536368                        0                             0   
536369                        0                         

## Step 2: Implement Apriori Algorithm
Step-by-Step Procedure:
1. Generate Frequent 1-Itemsets
Count the frequency (support) of each individual item in the dataset.
Keep only those with support ≥ min_support.
→ Result is L1 (frequent 1-itemsets)
2. Iterative Candidate Generation (k = 2 to n)
While L(k-1) is not empty:
a. Candidate Generation

Generate candidate itemsets Ck of size k from L(k-1) using the Apriori property:
Any (k-itemset) is only frequent if all of its (k−1)-subsets are frequent.
b. Prune Candidates
Eliminate candidates that have any (k−1)-subset not in L(k-1).
c. Count Support
For each transaction, count how many times each candidate in Ck appears.
d. Generate Frequent Itemsets
Form Lk by keeping candidates from Ck that meet the min_support.
Repeat until Lk becomes empty.
Implement the following functions:
1. `get_frequent_itemsets(transactions, min_support)` - Returns frequent itemsets and their support
2. `generate_candidates(prev_frequent_itemsets, k)` - Generates candidate itemsets of length `k`
3. `calculate_support(transactions, candidates)` - Calculates the support count for each candidate

**Write reusable functions** for each part of the algorithm.

In [12]:
# Implement apriori functions below
from itertools import combinations

def get_frequent_itemsets(transactions, min_support):
    # Your code here
    transactions = [set(t) for t in transactions]
    items = set(i for t in transactions for i in t)
    candidates = [frozenset([i]) for i in items]
    
    frequent_itemsets = {}
    k = 1
    
    while candidates:
        support_counts = calculate_support(transactions, candidates)
        Lk = {item: count for item, count in support_counts.items() if count >= min_support}
        
        if not Lk:
            break
        
        frequent_itemsets.update(Lk)
        candidates = generate_candidates(list(Lk.keys()), k+1)
        k += 1
    
    return frequent_itemsets
    pass

def generate_candidates(prev_frequent_itemsets, k):
    # Your code here
    candidates = set()
    prev_items = list(prev_frequent_itemsets)
    for i in range(len(prev_items)):
        for j in range(i+1, len(prev_items)):
            union = prev_items[i] | prev_items[j]
            if len(union) == k:
                candidates.add(union)
    return candidates
    pass

def calculate_support(transactions, candidates):
    # Your code here
    support_counts = {}
    for cand in candidates:
        count = 0
        for txn in transactions:
            if cand.issubset(txn):
                count += 1
        support_counts[cand] = count
    return support_counts
    pass




## Step 3: Generate Association Rules

- Use frequent itemsets to generate association rules
- For each rule `A => B`, calculate:
  - **Support**
  - **Confidence**
- Only return rules that meet a minimum confidence threshold (e.g., 0.5)

👉 **Implement rule generation function below**

In [15]:
from itertools import combinations

def generate_rules(frequent_itemsets, min_confidence, total_transactions):
    rules = []
    
    for itemset, support_count in frequent_itemsets.items():
        if len(itemset) < 2:
            continue  # need at least 2 items to form a rule
        
        # split itemset into all possible A,B
        for i in range(1, len(itemset)):
            for A in combinations(itemset, i):
                A = frozenset(A)
                B = itemset - A
                if not B:
                    continue

                # support of A∪B
                support = support_count / total_transactions

                # ensure A exists in frequent_itemsets
                if A in frequent_itemsets:
                    conf = support_count / frequent_itemsets[A]

                    if conf >= min_confidence:
                        rules.append((set(A), set(B), round(support, 3), round(conf, 3)))
    
    return rules


## Step 4: Output and Visualize

- Print top 10 frequent itemsets
- Print top 10 association rules (by confidence or lift)

👉 **Output results below**

In [ ]:
# Output the final results
# Optional: Add visualizations
# Parameters
min_support = 0.05
min_confidence = 0.5

# If your function expects transaction lists:
transactions = basket.apply(lambda row: [item for item in basket.columns if row[item] == 1], axis=1).tolist()
frequent_itemsets = get_frequent_itemsets(transactions, min_support)

# Generate rules
rules = generate_rules(frequent_itemsets, min_confidence)

# Display top 10 frequent itemsets
print("Top 10 Frequent Itemsets:")
for level in frequent_itemsets:
    for item, support in sorted(level.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"{set(item)} -> Support: {support:.2f}")

# Display top 10 rules
print("\nTop 10 Association Rules:")
for antecedent, consequent, support, confidence in rules[:10]:
    print(f"{set(antecedent)} => {set(consequent)} | Support: {support:.2f} | Confidence: {confidence:.2f}")

# Your Code Here